In [1]:
import time

import torch
from torch import nn

from pract11.Guia11Reformat.src.ClassifierMLP import ClassifierMLP
from pract11.Guia11Reformat.src.Config import Config
from pract11.Guia11Reformat.src.FashionMNISTData import FashionMNISTData
from pract11.Guia11Reformat.src.Optimizer import create_optimizer
from pract11.Guia11Reformat.src.Trainer import Trainer
from pract11.Guia11Reformat.src.Visualizer import Visualizer


def runExperiment(config, data):
    # Configuración
    device = config.setup()

    if config.show_sample:
        data.inspect_example()

    # Modelo
    torch.manual_seed(config.SEED)

    model = ClassifierMLP(
        p_dropout=config.DROPOUT_P, neuron_1=config.neuron_1, neuron_2=config.neuron_2,
    ).to(device)

    # Comprobar dimensiones
    if config.show_dimentions:
        images, labels = next(iter(data.train_loader))

        with torch.no_grad():
            classifications = model(images.to(device))

        print(f"images.shape: {images.shape}")
        print(f"labels.shape: {labels.shape}")

        n_params = sum(
            p.numel()
            for p in model.parameters()
            if p.requires_grad
        )
        print(f"Parámetros entrenables: {n_params:,}")

        print("\nDimensiones:")
        print("Entrada:", images.shape)
        print("Logits:", classifications.shape)

    # Función de pérdida y optimizador
    loss_fn = nn.CrossEntropyLoss()

    optimizer = create_optimizer(
        model=model,
        optimizer_name=config.OPTIMIZER_NAME,
        adam_lr=config.ADAM_LR,
        sgd_lr=config.SGD_LR
    )

    print("\nOptimizador:", config.OPTIMIZER_NAME)
    print("Dropout:", config.DROPOUT_P)

    # Untrained
    if config.show_untrained:
        Visualizer.showPredictions(model, device, data)

    # Entrenamiento
    trainer = Trainer(
        model,
        optimizer,
        loss_fn,
        device,
        config
    )
    start = time.perf_counter()
    trainer.fit(
        data.train_loader,
        data.valid_loader,
        epochs=config.EPOCHS
    )

    print(f"Training time: {time.perf_counter() - start:.2f} seconds")
    # Gráfico de pérdidas
    if config.show_loss: Visualizer.plot_history(trainer.history)

    # Evaluación final
    _, _, confusion_matrix = trainer.test(data.test_loader)

    #Trained
    if config.show_trained:
        Visualizer.showPredictions(model, device, data)
    return trainer, data, confusion_matrix


# ============================================================
# EJECUCIÓN
# ============================================================
# showSample=False, showDimentions=False, showUntrained=False, showLoss=False, showTrained=True, showDimentions2=False, showEpochData=True
# runExperiment(Config(dropout_p=0.2, epochs=2)


def pre_run(expies, data):
    results_together = {}
    for title, exp in expies.items():
        results = {}
        data.reset_generator()  # Reset before each experiment
        trainer, data, confusion_matrix = runExperiment(exp, data)
        results[title] = {"trainer": trainer, "data": data, "confusion_matrix": confusion_matrix}
        results_together[title] = {"trainer": trainer, "data": data, "confusion_matrix": confusion_matrix}
        Visualizer.showExpieResults(results)
    print("\n-----------------------------------Results TOGETHER:-----------------------------------\n")
    Visualizer.showExpieResults(results_together, show_confusion=False)
    Visualizer.showComparativeTable(results_together)
    return results_together


data_loaders = FashionMNISTData(
    batch_size=100,
    seed=42, show_dataset_info=False,
)


##TODO si uno quisiera hacer bien las cosas a nivel codigo, tendria estos tipos de test en una seccion de Tests, tambien tendria unit tests
### generator test ####
# data_loaders.reset_generator()
# batches_1 = data_loaders.get_first_batches()
#
# data_loaders.reset_generator()
# batches_2 = data_loaders.get_first_batches()
#
# for i, (b1, b2) in enumerate(zip(batches_1, batches_2)):
#     print(f"Batch {i}: {torch.equal(b1, b2)}") # me tiene que devolver True
### generator test ####

###model test####
# torch.manual_seed(42)
# model_1 = ClassifierMLP()
#
# torch.manual_seed(42)
# model_2 = ClassifierMLP()
#
# print(all(
#     torch.equal(p1, p2) # me tiene que devolver True
#     for p1, p2 in zip(model_1.parameters(), model_2.parameters())
# ))
##### model test ######

In [ ]:
expies_1 = {
    "SGD-0.001/128-64": Config(sgd_lr=0.001, dropout_p=0.2, epochs=30, optimizer_name="SGD", show_sample=False,
                               show_dimentions=False, neuron1=128, neuron2=64,
                               show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                               show_epoch_data=True, show_init=True, show_dataset_info=False),
    "SGD-0.01/128-64": Config(sgd_lr=0.01, dropout_p=0.2, epochs=30, optimizer_name="SGD", show_sample=False,
                              show_dimentions=False, neuron1=128, neuron2=64,
                              show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                              show_epoch_data=True, show_init=True, show_dataset_info=False),
    "SGD-0.1/128-64": Config(sgd_lr=0.1, dropout_p=0.2, epochs=30, optimizer_name="SGD", show_sample=False,
                             show_dimentions=False, neuron1=128, neuron2=64,
                             show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                             show_epoch_data=True, show_init=True, show_dataset_info=False),
    "Adam-0.0001/128-64": Config(adam_lr=0.0001, dropout_p=0.2, epochs=30, optimizer_name="Adam", show_sample=False,
                                 show_dimentions=False, neuron1=128, neuron2=64,
                                 show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                                 show_epoch_data=True, show_init=True, show_dataset_info=False),
    "Adam-0.001/128-64": Config(adam_lr=0.001, dropout_p=0.2, epochs=30, optimizer_name="Adam", show_sample=False,
                                show_dimentions=False, neuron1=128, neuron2=64,
                                show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                                show_epoch_data=True, show_init=True, show_dataset_info=False),
    "Adam-0.01/128-64": Config(adam_lr=0.01, dropout_p=0.2, epochs=30, optimizer_name="Adam", show_sample=False,
                               show_dimentions=False, neuron1=128, neuron2=64,
                               show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                               show_epoch_data=True, show_init=True, show_dataset_info=False)
}
results = pre_run(expies_1, data_loaders)


In [ ]:
expies_2 = {
    "SGD-0.001/64-32": Config(sgd_lr=0.001, dropout_p=0.2, epochs=30, optimizer_name="SGD", show_sample=False,
                              show_dimentions=False, neuron1=64, neuron2=32,
                              show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                              show_epoch_data=True, show_init=True, show_dataset_info=False),
    "SGD-0.01/64-32": Config(sgd_lr=0.01, dropout_p=0.2, epochs=30, optimizer_name="SGD", show_sample=False,
                             show_dimentions=False, neuron1=64, neuron2=32,
                             show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                             show_epoch_data=True, show_init=True, show_dataset_info=False),
    "SGD-0.1/64-32": Config(sgd_lr=0.1, dropout_p=0.2, epochs=30, optimizer_name="SGD", show_sample=False,
                            show_dimentions=False, neuron1=64, neuron2=32,
                            show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                            show_epoch_data=True, show_init=True, show_dataset_info=False),
    "Adam-0.0001/64-32": Config(adam_lr=0.0001, dropout_p=0.2, epochs=30, optimizer_name="Adam", show_sample=False,
                                show_dimentions=False, neuron1=64, neuron2=32,
                                show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                                show_epoch_data=True, show_init=True, show_dataset_info=False),
    "Adam-0.001/64-32": Config(adam_lr=0.001, dropout_p=0.2, epochs=30, optimizer_name="Adam", show_sample=False,
                               show_dimentions=False, neuron1=64, neuron2=32,
                               show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                               show_epoch_data=True, show_init=True, show_dataset_info=False),
    "Adam-0.01/64-32": Config(adam_lr=0.01, dropout_p=0.2, epochs=30, optimizer_name="Adam", show_sample=False,
                              show_dimentions=False, neuron1=64, neuron2=32,
                              show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                              show_epoch_data=True, show_init=True, show_dataset_info=False)
}

results2 = pre_run(expies_2, data_loaders)


In [ ]:
expies_3 = {
    "SGD-0.001/256-128": Config(sgd_lr=0.001, dropout_p=0.2, epochs=30, optimizer_name="SGD", show_sample=False,
                                show_dimentions=False, neuron1=256, neuron2=128,
                                show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                                show_epoch_data=True, show_init=True, show_dataset_info=False),
    "SGD-0.01/256-128": Config(sgd_lr=0.01, dropout_p=0.2, epochs=30, optimizer_name="SGD", show_sample=False,
                               show_dimentions=False, neuron1=256, neuron2=128,
                               show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                               show_epoch_data=True, show_init=True, show_dataset_info=False),
    "SGD-0.1/256-128": Config(sgd_lr=0.1, dropout_p=0.2, epochs=30, optimizer_name="SGD", show_sample=False,
                              show_dimentions=False, neuron1=256, neuron2=128,
                              show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                              show_epoch_data=True, show_init=True, show_dataset_info=False),
    "Adam-0.0001/256-128": Config(adam_lr=0.0001, dropout_p=0.2, epochs=30, optimizer_name="Adam", show_sample=False,
                                  show_dimentions=False, neuron1=256, neuron2=128,
                                  show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                                  show_epoch_data=True, show_init=True, show_dataset_info=False),
    "Adam-0.001/256-128": Config(adam_lr=0.001, dropout_p=0.2, epochs=30, optimizer_name="Adam", show_sample=False,
                                 show_dimentions=False, neuron1=256, neuron2=128,
                                 show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                                 show_epoch_data=True, show_init=True, show_dataset_info=False),
    "Adam-0.01/256-128": Config(adam_lr=0.01, dropout_p=0.2, epochs=30, optimizer_name="Adam", show_sample=False,
                                show_dimentions=False, neuron1=256, neuron2=128,
                                show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                                show_epoch_data=True, show_init=True, show_dataset_info=False)
}

results3 = pre_run(expies_3, data_loaders)